# Notebook 06 — Final Model Freeze + Final Test Evaluation

Freeze the verified Notebook 05C candidate:

- hidden_dim = 128
- dropout = 0.20
- learning_rate = 0.0005
- validation PR-AUC = 0.456421
- validation ROC-AUC = 0.792874
- validation F1 = 0.434164

The test set is evaluated **once, after freezing**. No tuning or model changes are allowed based on test results.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

ROOT=Path.cwd().resolve()
if ROOT.name.lower()=="notebooks": ROOT=ROOT.parent

GRAPH=ROOT/"artifacts/graph/bank_heterodata.pt"
META=ROOT/"artifacts/graph/graph_metadata.json"
CANDIDATE=ROOT/"artifacts/models/graphsage_focused_candidate.pt"
MODELS=ROOT/"artifacts/models"
RESULTS=ROOT/"artifacts/results"
MODELS.mkdir(parents=True,exist_ok=True)
RESULTS.mkdir(parents=True,exist_ok=True)

FINAL=MODELS/"graphsage_final.pt"
METRICS=RESULTS/"final_test_metrics.json"
CM=RESULTS/"final_confusion_matrix.csv"
PRED=RESULTS/"final_predictions.csv"
SUMMARY=RESULTS/"final_evaluation_summary.json"
PLOT=RESULTS/"final_confusion_matrix.png"

for p in [GRAPH,META,CANDIDATE]:
    if not p.exists(): raise FileNotFoundError(p)

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",DEVICE)


In [ ]:
# Load graph and frozen 05C candidate
try:
    data=torch.load(GRAPH,map_location="cpu",weights_only=False)
except TypeError:
    data=torch.load(GRAPH,map_location="cpu")

try:
    candidate=torch.load(CANDIDATE,map_location="cpu",weights_only=False)
except TypeError:
    candidate=torch.load(CANDIDATE,map_location="cpu")

with open(META,encoding="utf-8") as f: graph_metadata=json.load(f)

assert isinstance(data,HeteroData)
assert candidate["model_config"]["hidden_dim"]==128
assert abs(candidate["model_config"]["dropout"]-0.20)<1e-12
assert abs(candidate["model_config"]["learning_rate"]-0.0005)<1e-12

data=data.to(DEVICE)
train_mask=data["customer"].train_mask.bool()
val_mask=data["customer"].val_mask.bool()
test_mask=data["customer"].test_mask.bool()

assert torch.all(train_mask.to(torch.int8)+val_mask.to(torch.int8)+test_mask.to(torch.int8)==1)

print("Freeze contract passed.")
print("Validation metrics:",candidate["validation_metrics"])


## Frozen GraphSAGE Architecture

This exactly matches the architecture used for the selected 05C candidate. No optimizer or training loop is created in this notebook.


In [ ]:
class HeteroGraphSAGE(nn.Module):
    def __init__(self,metadata,hidden_dim=128,dropout=0.20):
        super().__init__()
        _,edges=metadata
        self.dropout=dropout
        self.conv1=HeteroConv({
            e:SAGEConv((-1,-1),hidden_dim,aggr="mean") for e in edges
        },aggr="sum")
        self.conv2=HeteroConv({
            e:SAGEConv((-1,-1),hidden_dim,aggr="mean") for e in edges
        },aggr="sum")
        self.classifier=nn.Linear(hidden_dim,1)

    def forward(self,x_dict,edge_index_dict):
        x=self.conv1(x_dict,edge_index_dict)
        x={k:F.dropout(F.relu(v),p=self.dropout,training=self.training) for k,v in x.items()}
        x=self.conv2(x,edge_index_dict)
        x={k:F.dropout(F.relu(v),p=self.dropout,training=self.training) for k,v in x.items()}
        return self.classifier(x["customer"]).squeeze(-1)

model=HeteroGraphSAGE(
    data.metadata(),
    hidden_dim=candidate["model_config"]["hidden_dim"],
    dropout=candidate["model_config"]["dropout"]
).to(DEVICE)
model.eval()

with torch.no_grad():
    model(data.x_dict,data.edge_index_dict)

model.load_state_dict(candidate["model_state_dict"])
model.eval()
print("Frozen candidate restored.")


## Pre-test reproducibility check

Reproduce the saved validation PR-AUC before accessing the test metrics.


In [ ]:
with torch.no_grad():
    logits=model(data.x_dict,data.edge_index_dict)

vp=torch.sigmoid(logits[val_mask]).cpu().numpy()
vy=data["customer"].y[val_mask].cpu().numpy()

reproduced_val_pr=average_precision_score(vy,vp)
reproduced_val_roc=roc_auc_score(vy,vp)
reproduced_val_f1=f1_score(vy,(vp>=0.5).astype(int),zero_division=0)

print("Saved validation PR-AUC:",candidate["validation_metrics"]["pr_auc"])
print("Reproduced validation PR-AUC:",reproduced_val_pr)
print("Reproduced validation ROC-AUC:",reproduced_val_roc)
print("Reproduced validation F1:",reproduced_val_f1)

assert abs(reproduced_val_pr-candidate["validation_metrics"]["pr_auc"])<1e-6
print("Validation reproducibility PASSED.")


# FINAL TEST EVALUATION

This is the final evaluation of the frozen model. The test set is not used for any subsequent model selection.


In [ ]:
with torch.no_grad():
    test_logits=model(data.x_dict,data.edge_index_dict)

test_prob=torch.sigmoid(test_logits[test_mask]).cpu().numpy()
test_y=data["customer"].y[test_mask].cpu().numpy()
test_pred=(test_prob>=0.5).astype(int)

final_metrics={
    "accuracy":float(accuracy_score(test_y,test_pred)),
    "precision":float(precision_score(test_y,test_pred,zero_division=0)),
    "recall":float(recall_score(test_y,test_pred,zero_division=0)),
    "f1":float(f1_score(test_y,test_pred,zero_division=0)),
    "roc_auc":float(roc_auc_score(test_y,test_prob)),
    "pr_auc":float(average_precision_score(test_y,test_prob))
}

print("="*80)
print("FINAL TEST RESULTS")
for k,v in final_metrics.items(): print(f"{k.upper():10s}: {v:.6f}")
print("="*80)
print(classification_report(test_y,test_pred,target_names=["no","yes"],zero_division=0,digits=4))


In [ ]:
# Confusion matrix
cm=confusion_matrix(test_y,test_pred)
cm_df=pd.DataFrame(cm,index=["Actual no","Actual yes"],columns=["Predicted no","Predicted yes"])
display(cm_df)
cm_df.to_csv(CM)

plt.figure(figsize=(6,5))
plt.imshow(cm,interpolation="nearest")
plt.title("Final Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0,1],["no","yes"]); plt.yticks([0,1],["no","yes"])
for i in range(2):
    for j in range(2): plt.text(j,i,str(cm[i,j]),ha="center",va="center")
plt.tight_layout()
plt.savefig(PLOT,dpi=150,bbox_inches="tight")
plt.show()


In [ ]:
# Save test predictions
indices=torch.where(test_mask)[0].cpu().numpy()
pred_df=pd.DataFrame({
    "customer_index":indices,
    "y_true":test_y.astype(int),
    "predicted_probability":test_prob,
    "y_pred":test_pred.astype(int)
})
pred_df.to_csv(PRED,index=False)

# Freeze as final production artifact
final_checkpoint={
    "model_state_dict":candidate["model_state_dict"],
    "model_config":candidate["model_config"],
    "best_epoch":candidate["best_epoch"],
    "validation_metrics":candidate["validation_metrics"],
    "test_metrics":final_metrics,
    "graph_metadata":graph_metadata,
    "freeze_status":"FINAL_FROZEN_MODEL",
    "classification_threshold":0.5
}
torch.save(final_checkpoint,FINAL)

summary={
    "model_status":"FINAL_FROZEN_MODEL",
    "source_candidate":str(CANDIDATE.relative_to(ROOT)),
    "final_model":str(FINAL.relative_to(ROOT)),
    "validation_metrics":candidate["validation_metrics"],
    "test_metrics":final_metrics,
    "classification_threshold":0.5,
    "test_sample_count":int(test_mask.sum()),
    "further_tuning_after_test":False
}
with open(METRICS,"w") as f: json.dump(final_metrics,f,indent=2)
with open(SUMMARY,"w") as f: json.dump(summary,f,indent=2)

print("Saved:",FINAL)
print("Saved:",METRICS)
print("Saved:",PRED)


## Final model reload verification

Reload the frozen checkpoint and verify that it reproduces the final test probabilities.


In [ ]:
try:
    saved=torch.load(FINAL,map_location=DEVICE,weights_only=False)
except TypeError:
    saved=torch.load(FINAL,map_location=DEVICE)

reloaded=HeteroGraphSAGE(
    data.metadata(),
    hidden_dim=saved["model_config"]["hidden_dim"],
    dropout=saved["model_config"]["dropout"]
).to(DEVICE)
reloaded.eval()
with torch.no_grad(): reloaded(data.x_dict,data.edge_index_dict)
reloaded.load_state_dict(saved["model_state_dict"])
reloaded.eval()

with torch.no_grad():
    reload_logits=reloaded(data.x_dict,data.edge_index_dict)
reload_prob=torch.sigmoid(reload_logits[test_mask]).cpu().numpy()

assert np.allclose(test_prob,reload_prob,atol=1e-7)
assert saved["freeze_status"]=="FINAL_FROZEN_MODEL"

for p in [FINAL,METRICS,CM,PRED,SUMMARY,PLOT]:
    assert p.exists(),p

print("="*85)
print("NOTEBOOK 06 VERIFICATION PASSED")
print("="*85)
print("FINAL FROZEN MODEL")
print("Validation PR-AUC:",f"{candidate['validation_metrics']['pr_auc']:.6f}")
print("Test PR-AUC:",f"{final_metrics['pr_auc']:.6f}")
print("Test ROC-AUC:",f"{final_metrics['roc_auc']:.6f}")
print("Test F1:",f"{final_metrics['f1']:.6f}")
print("Test Precision:",f"{final_metrics['precision']:.6f}")
print("Test Recall:",f"{final_metrics['recall']:.6f}")
print("Test samples:",int(test_mask.sum()))
print("Final model:",FINAL)
print("="*85)
print("STOP: model is frozen. Do not tune using test results.")
